# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema hosted at a public URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print a summary from metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Dataset Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all available record sets (by @id, name, and short description)
print("Available Record Sets:")
record_sets = []

for rs in getattr(metadata, 'recordSets', []):
    # Each rs is a RecordSet object with an @id, name, and fields
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', 'No name')
    rs_desc = getattr(rs, 'description', 'No description')
    record_sets.append(rs_id)
    print(f"- @id: {rs_id} | name: {rs_name} | description: {rs_desc}")

# If there are no recordSets attribute, fallback to a single default
if not record_sets:
    # For some schemas, you can list distributions as tabular sources.
    if hasattr(metadata, 'distribution'):
        from collections.abc import Iterable
        distributions = metadata.distribution
        # Distribution(s) can be a list of objects with an '@id' field
        if not isinstance(distributions, Iterable) or isinstance(distributions, str):
            distributions = [distributions]
        for d in distributions:
            d_id = getattr(d, '@id', str(d))
            record_sets.append(d_id)
        print("\n[Info] No explicit Croissant record sets found, using distribution IDs as record sets:")
        for d_id in record_sets:
            print(f"- @id: {d_id}")

# Choose the primary record set for this dataset
record_set_id = None
if record_sets:
    record_set_id = record_sets[0]
    print(f"\nDefault record set selected for loading: {record_set_id}")

# Show available fields for the default record set (by their @id and name)
print("\nFields in the default record set:")
fields = []
if hasattr(metadata, 'recordSets') and metadata.recordSets:
    main_rs = metadata.recordSets[0]
    for field in getattr(main_rs, 'fields', []):
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', 'No name')
        print(f"- @id: {field_id} | name: {field_name}")
        fields.append(field_id)
elif hasattr(metadata, 'fields'):
    for field in metadata.fields:
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', 'No name')
        print(f"- @id: {field_id} | name: {field_name}")
        fields.append(field_id)

## 3. Data Extraction
Load data from the record set into a DataFrame for analysis. We refer to record set and field `@id`s as identified in the overview.

In [ ]:
# Extract all records from each available record set by their @id
dataframes = {}

for rs_id in record_sets:
    # Iterate through records and collect as dicts
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set @id: {rs_id}")
    except Exception as e:
        print(f"Could not load records from {rs_id}: {e}")

# For demonstration, pick the first loaded DataFrame
main_df_key = None
if dataframes:
    main_df_key = list(dataframes.keys())[0]
    print(f"\nFields (columns) for {main_df_key} (showing their @id):\n", dataframes[main_df_key].columns.tolist())
    display(dataframes[main_df_key].head())
else:
    print("No DataFrame could be created.")

## 4. Exploratory Data Analysis (EDA)
Let's process the loaded DataFrame: filter numeric fields (by their `@id`), normalize them, and group by a categorical field, all referencing fields by their `@id`.

In [ ]:
# Select a numeric field by @id (change as appropriate for your dataset)
# You can choose the actual field IDs from the field list in Section 2. Adjust below if needed.
numeric_field_id = None
group_field_id = None
if main_df_key is not None:
    df = dataframes[main_df_key]
    # Try to auto-select likely numeric and group fields by looking for common keywords
    for col in df.columns:
        if any(word in col.lower() for word in ['age', 'value', 'interval', 'count', 'number']):
            numeric_field_id = col
            break
    # Pick a field for grouping (e.g., 'Sex', 'msi', 'anatomical_location')
    for col in df.columns:
        if any(word in col.lower() for word in ['sex', 'location', 'status', 'msi', 'group']):
            group_field_id = col
            break

    if numeric_field_id:
        print(f"Selected numeric field for EDA: {numeric_field_id}")
        # Remove any clearly erroneous or non-numeric entries, coerce to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Filter: for example, show records where the value is >10 (arbitrary threshold)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field_id if possible
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count', 'min', 'max'])
            print(f"Grouped statistics for {numeric_field_id} by {group_field_id}:")
            display(grouped_df)
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No main DataFrame available for EDA.")

## 5. Visualization
Visualize key distributions or relationships in the data. Here we show distributions for the chosen numeric field and relationships with a grouping field, using `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df_key is not None and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(data=dataframes[main_df_key], x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group (if possible)
    if group_field_id and group_field_id in dataframes[main_df_key].columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(
            data=dataframes[main_df_key], x=group_field_id, y=numeric_field_id
        )
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, you learned to load and explore the FAIR^2 clinical oncology dataset via its Croissant schema using the `mlcroissant` library. By referencing every data structure through its `@id`, we've ensured reliable and reproducible data access. You performed exploratory data analysis and visualized field distributions. For further research, consider deeper feature engineering, integrating clinical ontologies, or building predictive models for clinicopathological outcomes.